# Alabama — Title 27 (Insurance) → `data/alabama/ins_codes/*.md`

Alison’s **HTML statute pages** are a SPA shell; the legislature exposes the Code through a **public GraphQL** endpoint. This notebook:

1. Calls `codesOfAlabama(offset: …)` for the **two** 2,500-row pages that contain **Title 27** sections (`Section 27-…`).
2. Keeps rows whose `shortTitle` starts with **`Section 27-`**.
3. Converts `content` HTML to plain text and writes one **Markdown** file per section under **`data/alabama/ins_codes/`** (prefix `ALA_sec_`).

**Source:** [Alabama Legislature — Alison](https://alison.legislature.state.al.us/) — data from `POST …/graphql` (`codesOfAlabama`).

**Politeness:** default **0.25 s** pause between GraphQL calls (two heavy requests with `content`). Increase if needed.

Then run **`python -m app.ingest`** from the project root (or Re-index in the UI).

In [1]:
%pip install -q httpx beautifulsoup4

You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import json
import re
import time
from pathlib import Path

import httpx
from bs4 import BeautifulSoup

GRAPHQL_URL = "https://alison.legislature.state.al.us/graphql"
# Only these offsets contain Title 27 "Section 27-*" rows (2500 rows per page).
OFFINS_OFFSETS = (25000, 27500)

OUT_DIR = Path("data") / "alabama" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "RAG-AL-Title27/1.0 (public Code of Alabama; educational indexing)"
REQUEST_DELAY_SEC = 0.25
TIMEOUT = 120.0

SECTION_PREFIX = re.compile(r"^Section\s+27-", re.I)

In [3]:
def graphql(payload: dict) -> dict:
    with httpx.Client(
        headers={
            "User-Agent": USER_AGENT,
            "Content-Type": "application/json",
        },
        timeout=TIMEOUT,
        http2=False,
    ) as client:
        r = client.post(GRAPHQL_URL, json=payload)
        r.raise_for_status()
        time.sleep(REQUEST_DELAY_SEC)
        return r.json()


def fetch_page_with_content(offset: int) -> list[dict]:
    q = """
    query Page($offset: Int!) {
      codesOfAlabama(offset: $offset) {
        count
        data {
          id
          codeId
          shortTitle
          title
          content
        }
      }
    }
    """
    j = graphql({"query": q, "variables": {"offset": offset}})
    if j.get("errors"):
        raise RuntimeError(j["errors"])
    block = j["data"]["codesOfAlabama"]
    print(f"offset={offset} total_count={block['count']} rows={len(block['data'])}")
    return block["data"]


def html_to_text(html: str | None) -> str:
    if not html:
        return ""
    return BeautifulSoup(html, "html.parser").get_text("\n", strip=True)


def short_title_to_file_key(short_title: str) -> str:
    # "Section 27-1-1" -> "27-1-1"
    m = re.match(r"^Section\s+(.+)$", short_title.strip(), re.I)
    key = (m.group(1) if m else short_title).strip()
    return re.sub(r"[^0-9A-Za-z.-]+", "_", key)


def official_url(code_id: str) -> str:
    return f"https://alison.legislature.state.al.us/alison/codeofalabama/1975/{code_id}"


def download_title27() -> int:
    rows_out: list[dict] = []
    for off in OFFINS_OFFSETS:
        for row in fetch_page_with_content(off):
            st = (row.get("shortTitle") or "").strip()
            if SECTION_PREFIX.match(st):
                rows_out.append(row)
    print(f"Title 27 sections found: {len(rows_out)}")
    wrote = 0
    for row in rows_out:
        st = row["shortTitle"].strip()
        title_line = (row.get("title") or st).strip()
        body = html_to_text(row.get("content"))
        code_id = str(row.get("codeId") or "")
        src = official_url(code_id)
        fn = f"ALA_sec_{short_title_to_file_key(st)}.md"
        dest = OUT_DIR / fn
        md = (
            f"# {title_line}\n\n"
            f"**Code of Alabama — Title 27 (Insurance)**\n\n"
            f"**Official source (Alison):** {src}\n\n"
            f"**GraphQL id:** `{row.get('id')}`\n\n"
            f"---\n\n"
            f"{body}\n"
        )
        dest.write_text(md, encoding="utf-8")
        wrote += 1
    (OUT_DIR / "_alabama_title27_keys.txt").write_text(
        "\n".join(r["shortTitle"] for r in rows_out),
        encoding="utf-8",
    )
    print(f"Wrote {wrote} files to {OUT_DIR.resolve()}")
    return wrote


download_title27()

offset=25000 total_count=58096 rows=2500
offset=27500 total_count=58096 rows=2500
Title 27 sections found: 1533
Wrote 1533 files to /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/alabama/ins_codes


1533

## Next step

Run **`python -m app.ingest`** from the repository root so `data/alabama/ins_codes/*.md` is embedded in your vector store.